# 02 - NRI Data Pipeline

Takes the long-format NRI data produced by `01_nri_explore.ipynb` and prepares it for joining to the analysis dataset.

**Input:** `NRI_Long.csv` — one row per county per NRI vintage (2020, 2021, 2023, 2025)

**Output:** `nri_panel.parquet` — one row per county per storm year (2020–2025), with NRI scores vintage-matched to each year

**Vintage matching rule (biennial release schedule):**
- 2020 → NRI 2020
- 2021 → NRI 2021
- 2022 → NRI 2021 (forward-fill, no 2022 release)
- 2023 → NRI 2023
- 2024 → NRI 2023 (forward-fill, no 2024 release)
- 2025 → NRI 2025

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Load

In [2]:
nri_raw = pd.read_csv('../data/processed/NRI_Long.csv', index_col=0)

print(f'Shape: {nri_raw.shape}')
print(f'Vintages: {sorted(nri_raw["year"].unique())}')
print(f'Unique counties: {nri_raw["nri_id"].nunique()}')
nri_raw.head()

Shape: (12579, 9)
Vintages: [2020, 2021, 2023, 2025]
Unique counties: 3145


,nri_id,stateabbrv,county,risk_value,eal_valt,sovi_score,resl_score,crf_value,year
0,C01001,AL,Autauga,2.602012e+06,3.311627e+06,25.857312,55.529800,0.785720,2020
1,C01001,AL,Autauga,2.087423e+06,2.656700e+06,25.857312,55.529800,0.785720,2021
2,C01001,AL,Autauga,6.156054e+06,5.514047e+06,51.299999,51.810001,1.116431,2023
3,C01001,AL,Autauga,2.051090e+07,1.975657e+07,38.040712,55.120865,1.038181,2025
4,C01003,AL,Baldwin,1.834143e+07,1.816858e+07,34.292471,54.759202,1.009514,2020


## Clean and derive FIPS

In [3]:
nri = nri_raw.copy()

# Derive 5-digit FIPS from nri_id (format: 'C' + FIPS)
nri['stcofips'] = nri['nri_id'].str[1:]

# Parse into state and county FIPS components for joining
nri['state_fips'] = nri['stcofips'].str[:2]
nri['county_fips'] = nri['stcofips'].str[2:]

# Rename year to nri_vintage to distinguish from storm year
nri = nri.rename(columns={'year': 'nri_vintage'})

# Drop redundant index column
nri = nri.drop(columns=['nri_id'])

print(nri.dtypes)
nri.head()

stateabbrv      object
county          object
risk_value     float64
eal_valt       float64
sovi_score     float64
resl_score     float64
crf_value      float64
nri_vintage      int64
stcofips        object
state_fips      object
county_fips     object
dtype: object


,stateabbrv,county,risk_value,eal_valt,sovi_score,resl_score,crf_value,nri_vintage,stcofips,state_fips,county_fips
0,AL,Autauga,2.602012e+06,3.311627e+06,25.857312,55.529800,0.785720,2020,01001,01,001
1,AL,Autauga,2.087423e+06,2.656700e+06,25.857312,55.529800,0.785720,2021,01001,01,001
2,AL,Autauga,6.156054e+06,5.514047e+06,51.299999,51.810001,1.116431,2023,01001,01,001
3,AL,Autauga,2.051090e+07,1.975657e+07,38.040712,55.120865,1.038181,2025,01001,01,001
4,AL,Baldwin,1.834143e+07,1.816858e+07,34.292471,54.759202,1.009514,2020,01003,01,003


## Validate — no nulls, FIPS formatting correct

In [4]:
assert nri.isnull().sum().sum() == 0, 'Unexpected nulls in NRI data'
assert nri['stcofips'].str.len().eq(5).all(), 'FIPS not all 5 digits'
assert nri['state_fips'].str.len().eq(2).all(), 'State FIPS not all 2 digits'
assert nri['county_fips'].str.len().eq(3).all(), 'County FIPS not all 3 digits'
assert set(nri['nri_vintage'].unique()) == {2020, 2021, 2023, 2025}, 'Unexpected vintages'

print('All assertions passed')

All assertions passed


## Expand to storm years via vintage matching

Cross-join each county's vintages against the storm years (2020–2025),
then assign the correct NRI vintage to each storm year.

In [5]:
# Vintage mapping: storm_year -> nri_vintage
VINTAGE_MAP = {
    2020: 2020,
    2021: 2021,
    2022: 2021,  # forward-fill, no 2022 release
    2023: 2023,
    2024: 2023,  # forward-fill, no 2024 release
    2025: 2025,
}

storm_years = pd.DataFrame({'storm_year': list(VINTAGE_MAP.keys())})
storm_years['nri_vintage'] = storm_years['storm_year'].map(VINTAGE_MAP)

# Get unique county identifiers
county_ids = nri[['stcofips', 'state_fips', 'county_fips', 'stateabbrv', 'county']].drop_duplicates(subset='stcofips')

# Cross join counties x storm years
county_years = county_ids.merge(storm_years, how='cross')

# Join NRI scores by county + vintage
nri_scores = nri.drop(columns=['stateabbrv', 'county', 'state_fips', 'county_fips'])

nri_panel = county_years.merge(
    nri_scores,
    on=['stcofips', 'nri_vintage'],
    how='left'
)

print(f'Panel shape: {nri_panel.shape}')
print(f'Expected rows: {county_ids.shape[0]} counties x 6 years = {county_ids.shape[0] * 6}')
nri_panel.head(12)

Panel shape: (18870, 12)
Expected rows: 3145 counties x 6 years = 18870


,stcofips,state_fips,county_fips,stateabbrv,county,storm_year,nri_vintage,risk_value,eal_valt,sovi_score,resl_score,crf_value
0,01001,01,001,AL,Autauga,2020,2020,2.602012e+06,3.311627e+06,25.857312,55.529800,0.785720
1,01001,01,001,AL,Autauga,2021,2021,2.087423e+06,2.656700e+06,25.857312,55.529800,0.785720
2,01001,01,001,AL,Autauga,2022,2021,2.087423e+06,2.656700e+06,25.857312,55.529800,0.785720
3,01001,01,001,AL,Autauga,2023,2023,6.156054e+06,5.514047e+06,51.299999,51.810001,1.116431
4,01001,01,001,AL,Autauga,2024,2023,6.156054e+06,5.514047e+06,51.299999,51.810001,1.116431
5,01001,01,001,AL,Autauga,2025,2025,2.051090e+07,1.975657e+07,38.040712,55.120865,1.038181
6,01003,01,003,AL,Baldwin,2020,2020,1.834143e+07,1.816858e+07,34.292471,54.759202,1.009514
7,01003,01,003,AL,Baldwin,2021,2021,1.190492e+07,1.179273e+07,34.292471,54.759202,1.009514
8,01003,01,003,AL,Baldwin,2022,2021,1.190492e+07,1.179273e+07,34.292471,54.759202,1.009514
9,01003,01,003,AL,Baldwin,2023,2023,2.106327e+08,2.020151e+08,31.030001,86.120003,1.042658


## Handle missing vintages and validate panel

**Known edge case:** CT Tolland county (FIPS 09013) has no 2025 NRI record because Connecticut
replaced its 8 legacy counties with 9 planning regions in 2024. We forward-fill from its 2023
vintage, consistent with the forward-fill logic applied to all counties for 2022 and 2024.

In [6]:
score_cols = ['risk_value', 'eal_valt', 'sovi_score', 'resl_score', 'crf_value']
missing_mask = nri_panel['resl_score'].isnull()
fips_missing = nri_panel.loc[missing_mask, 'stcofips'].unique()

if len(fips_missing) > 0:
    print(f'Forward-filling {len(fips_missing)} county-vintage(s) from 2023:')
    print(nri_panel.loc[missing_mask, ['stcofips', 'stateabbrv', 'county', 'storm_year']])
    for fips in fips_missing:
        src_rows = nri_panel[(nri_panel['stcofips'] == fips) & (nri_panel['nri_vintage'] == 2023)]
        tgt_idx = nri_panel.index[(nri_panel['stcofips'] == fips) & missing_mask]
        for col in score_cols:
            nri_panel.loc[tgt_idx, col] = src_rows[col].values[0]

assert nri_panel.isnull().sum().sum() == 0, 'Nulls remain after fill'
assert nri_panel.duplicated(['stcofips', 'storm_year']).sum() == 0, 'Duplicate county-year rows'
assert set(nri_panel['storm_year'].unique()) == set(VINTAGE_MAP.keys()), 'Missing storm years'

for storm_year, expected_vintage in VINTAGE_MAP.items():
    actual = nri_panel.loc[nri_panel['storm_year'] == storm_year, 'nri_vintage'].unique()
    assert list(actual) == [expected_vintage], f'Wrong vintage for {storm_year}: got {actual}'

print('All panel assertions passed')
print(f'\nVintage assignments:')
print(nri_panel.groupby('storm_year')['nri_vintage'].first().reset_index())

Forward-filling 1 county-vintage(s) from 2023:
     stcofips stateabbrv   county  storm_year
1859    09013         CT  Tolland        2025
All panel assertions passed

Vintage assignments:
   storm_year  nri_vintage
0        2020         2020
1        2021         2021
2        2022         2021
3        2023         2023
4        2024         2023
5        2025         2025


## Summary statistics by vintage

In [7]:
nri_panel.groupby('nri_vintage')[['resl_score', 'risk_value', 'eal_valt', 'sovi_score']].describe().round(2)

resl_score                                                    \
                 count   mean    std    min    25%    50%    75%     max   
nri_vintage                                                                
2020            3145.0  54.59   2.94  41.19  52.65  54.66  56.75   64.67   
2021            6290.0  54.59   2.94  41.19  52.65  54.66  56.75   64.67   
2023            6290.0  50.02  28.88   0.00  25.02  50.03  75.02  100.00   
2025            3145.0  50.02  28.87   0.03  25.03  50.03  75.03  100.00   

            risk_value               ...     eal_valt                \
                 count         mean  ...          75%           max   
nri_vintage                          ...                              
2020            3145.0  11953304.54  ...   8736234.26  1.406614e+09   
2021            6290.0  11431829.89  ...   7084387.29  1.433566e+09   
2023            6290.0  27884875.67  ...  13111339.12  3.916212e+09   
2025            3145.0  51487222.43  ...  32502368.70  7.601846e+09   

            sovi_score                                                   
                 count   mean    std    min    25%    50%    75%    max  
nri_vintage                                                              
2020            3145.0  38.34  11.15   0.01  31.85  38.31  44.46  100.0  
2021            6290.0  38.34  11.15   0.01  31.85  38.31  44.46  100.0  
2023            6290.0  50.01  28.87   0.00  25.02  50.03  74.98  100.0  
2025            3145.0  50.61  28.00  10.62  25.00  50.03  75.03  100.0  

[4 rows x 32 columns]

## Export

In [9]:
output_cols = [
    'stcofips', 'state_fips', 'county_fips', 'stateabbrv', 'county',
    'storm_year', 'nri_vintage',
    'resl_score', 'risk_value', 'eal_valt', 'sovi_score', 'crf_value'
]

nri_panel = nri_panel[output_cols].sort_values(['stcofips', 'storm_year']).reset_index(drop=True)

nri_panel.to_parquet('../data/processed/nri_panel.parquet', index=False)

print(f'Exported nri_panel.parquet')
print(f'Shape: {nri_panel.shape}')
print(f'Columns: {nri_panel.columns.tolist()}')

ArrowKeyError: No type extension with name arrow.py_extension_type found